In [30]:
import warnings
warnings.filterwarnings('ignore')

In [31]:
import torch
import torch.nn as nn

In [44]:
data = [
    ("I love machine", "learning"),
    ("I love deep", "learning"),
    ("I like machine", "learning"),
    ("I like deep", "learning"),
    ("machine learning is", "powerful"),
    ("deep learning is", "powerful"),
    ("Python is very", "useful"),
    ("PyTorch is very", "useful"),
    ("AI is curcial for out", "task"),
    ("GRU is very", "powerful"),
]

In [45]:
def build_vocab(sentences):
    vocab = {
        "<pad>": 0,
        "<unk>": 1
    }

    for sentence in sentences:

        for word in sentence.split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

input_vocab = build_vocab(x[0] for x in data)
target_vocab = build_vocab(x[1] for x in data)

In [46]:
print(input_vocab)
print(target_vocab)

{'<pad>': 0, '<unk>': 1, 'I': 2, 'love': 3, 'machine': 4, 'deep': 5, 'like': 6, 'learning': 7, 'is': 8, 'Python': 9, 'very': 10, 'PyTorch': 11, 'AI': 12, 'curcial': 13, 'for': 14, 'out': 15, 'GRU': 16}
{'<pad>': 0, '<unk>': 1, 'learning': 2, 'powerful': 3, 'useful': 4, 'task': 5}


In [47]:
def text_to_number(sentence, vocab):
    return torch.tensor(
        [vocab.get(word, vocab['<unk>']) for word in sentence.split()],
        dtype=torch.long
    )

input = [text_to_number(sentence, input_vocab) for sentence, _ in data]
output = [text_to_number(sentence, target_vocab) for _, sentence in data]

In [48]:
print(input), print(output)

[tensor([2, 3, 4]), tensor([2, 3, 5]), tensor([2, 6, 4]), tensor([2, 6, 5]), tensor([4, 7, 8]), tensor([5, 7, 8]), tensor([ 9,  8, 10]), tensor([11,  8, 10]), tensor([12,  8, 13, 14, 15]), tensor([16,  8, 10])]
[tensor([2]), tensor([2]), tensor([2]), tensor([2]), tensor([3]), tensor([3]), tensor([4]), tensor([4]), tensor([5]), tensor([3])]


(None, None)

In [49]:
from torch.nn.utils.rnn import pad_sequence

input_pad = pad_sequence(
    input,
    batch_first=True,
    padding_value=input_vocab["<pad>"]
)

print(input_pad)

tensor([[ 2,  3,  4,  0,  0],
        [ 2,  3,  5,  0,  0],
        [ 2,  6,  4,  0,  0],
        [ 2,  6,  5,  0,  0],
        [ 4,  7,  8,  0,  0],
        [ 5,  7,  8,  0,  0],
        [ 9,  8, 10,  0,  0],
        [11,  8, 10,  0,  0],
        [12,  8, 13, 14, 15],
        [16,  8, 10,  0,  0]])


In [53]:
vocab_size = len(input_vocab)
embedding_dim = 10

embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim
)

In [54]:
x = torch.tensor([2, 3, 4])
embedded = embedding(x)
print(embedded)

tensor([[ 0.1433,  0.4621, -1.5184,  0.6614,  0.9563, -0.7032,  0.5145,  0.1287,
         -0.1454, -1.9295],
        [ 0.0325, -1.2498,  0.2519, -1.6553, -0.4488,  0.2549,  1.2215,  0.9655,
          0.0933, -1.2068],
        [-1.1606, -0.3003, -0.4396,  0.5604, -0.5138,  0.3047,  1.0688, -1.7896,
          0.3669,  0.9408]], grad_fn=<EmbeddingBackward0>)


In [59]:
class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
        )

        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim
        )

    def forward(self, sentence):
        embedded = self.embedding(sentence)
        return self.gru(embedded)

In [60]:
model = GRUClassifier(vocab_size, embedding_dim, 32)

In [64]:
x = model(input_pad)

In [65]:
print(len(x))
print(type(x))

2
<class 'tuple'>


In [68]:
output, hidden = x
print(output.shape)

torch.Size([10, 5, 32])


In [69]:
print(hidden)

tensor([[[-0.0691, -0.2480, -0.0123, -0.0262, -0.4053,  0.3711, -0.0994,
          -0.1437, -0.2158, -0.2671, -0.1436,  0.1136,  0.2417, -0.0132,
           0.0243,  0.1827, -0.0373, -0.2134, -0.2746, -0.0882,  0.0561,
          -0.2980,  0.0704,  0.1486,  0.1653, -0.3131,  0.2049,  0.3093,
          -0.0209,  0.1161,  0.0665,  0.0296],
         [ 0.3157,  0.1848,  0.3121,  0.2512, -0.2059,  0.3686, -0.7613,
           0.3546,  0.0373, -0.7077, -0.4075, -0.6233,  0.0752, -0.0239,
           0.0011, -0.1574,  0.3428, -0.1292, -0.1960,  0.1487,  0.6670,
           0.2846, -0.0974,  0.1121, -0.2746, -0.5676, -0.1970,  0.2255,
          -0.4452,  0.2182, -0.1025, -0.2688],
         [ 0.1404,  0.2025,  0.2793, -0.2265, -0.4063, -0.3595,  0.1493,
          -0.4250, -0.1799, -0.1729,  0.0565, -0.0716, -0.4313,  0.0313,
           0.2638, -0.0095, -0.0423, -0.1650,  0.2001, -0.3658,  0.1305,
           0.0764, -0.1260,  0.0490, -0.3265,  0.1481, -0.3069,  0.3935,
           0.1891, -0.3633,  0